In [ ]:
#IMPORT LIBRARIES
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns

In [ ]:
#LOAD DATA
df=pd.read_csv("deliveries.csv")
df.head(5)

In [ ]:
#Data Exploration
df.columns
df.info
#shifting data into another variable
data=df

In [ ]:
#checking null values
data.isnull().sum()

In [ ]:
#Data Cleaning & Handling Missing Values
data["extras_type"]=data["extras_type"].fillna("no_extra")

data["player_dismissed"].value_counts()
data["player_dismissed"]=data["player_dismissed"].fillna("not_out")
data["dismissal_kind"]=data["dismissal_kind"].fillna("no_dismissle")
data["fielder"]=data["fielder"].fillna("no_fielder")

In [ ]:
data.isnull().sum()

In [ ]:
data.head()

In [ ]:
# Encoding Categorical Features
data=pd.get_dummies(df,columns=["batting_team","bowling_team","extras_type","player_dismissed","fielder"],drop_first=True,dtype=int)
data.head(5)


In [ ]:
#Batting Analysis (top batter by runs)
top_batters = data.groupby('batter')['batsman_runs'].sum().sort_values(ascending=False).head(10)
top_batters = top_batters.reset_index()
top_batters

In [ ]:
#VISULISING TOP BATTERES
plt.figure(figsize=(12,6))
sns.barplot(
    x=top_batters["batter"],
    y=top_batters["batsman_runs"],
    palette="Set1"
    )
plt.title("Leading Run Scorers in IPL")
plt.xticks(rotation=45)
plt.xlabel("Batters")
plt.ylabel("Total Runs")
plt.tight_layout()
plt.savefig("top_batters.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
#Most 50s and 100s
batsman_run_by_match=data.groupby(["batter","match_id"])["batsman_runs"].sum().reset_index()
batsman_run_by_match["100"]=batsman_run_by_match["batsman_runs"].apply(lambda x: 1 if x>=100 else 0)
batsman_run_by_match["50"]=batsman_run_by_match["batsman_runs"].apply(lambda x: 1 if 50<=x<100 else 0)

most_century_AND_halfCentury=batsman_run_by_match.groupby("batter")[["100","50"]].sum().sort_values(by="100",ascending=False).reset_index().head(10)
most_century_AND_halfCentury

In [ ]:
#Visualization Most 50s and 100s
plt.figure(figsize=(14,5))
# first graph
plt.subplot(1,2,1)
sns.barplot(
    data=most_century_AND_halfCentury,
    x="batter",
    y="100",
    palette="Set1"
)
plt.title("100s")
plt.xticks(rotation=45)

# second graph
plt.subplot(1,2,2)
sns.barplot(
    data=most_century_AND_halfCentury,
    x="batter",
    y="50",
    palette="Set2"
)
plt.title("50s")
plt.xticks(rotation=45)

plt.tight_layout()
plt.savefig("most_50_100s.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
#Strike Rate
total_ball_faced=data.groupby("batter")["ball"].count().reset_index()
total_ball_faced
total_run=data.groupby("batter")["batsman_runs"].sum().reset_index()
total_run
batsman_stats=total_run.merge(total_ball_faced,on="batter")
batsman_stats["strike_rate"]=(batsman_stats["batsman_runs"]/batsman_stats["ball"])*100
batsman_stats=batsman_stats.sort_values(by="strike_rate",ascending=False).head(10).reset_index(drop=True).sort_values(by="strike_rate",ascending=False)
batsman_stats

In [ ]:
#Visualization strike rate
plt.figure(figsize=(12,6))
sns.barplot(
    data=batsman_stats,
    x="batter",
    y="strike_rate",
    palette="Set1"
)

plt.title("Top 10 Batsmen by Strike Rate")
plt.xticks(rotation=45)
plt.xlabel("Batters")
plt.ylabel("Strike Rates")

plt.tight_layout()
plt.savefig("strike_rate.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
#Bowling Analysis (Top Bowlers by Wickets)
top_bowlers=data[data['is_wicket']==1].groupby('bowler')['is_wicket'].sum().sort_values(ascending=False).head(10)
top_bowlers=top_bowlers.reset_index().sort_values(by="is_wicket",ascending=False)
top_bowlers

In [ ]:
# Visualization(Bowling Analysis (Top Bowlers by Wickets))
plt.figure(figsize=(10,6))
sns.barplot(
    data=top_bowlers,
    x="bowler",
    y="is_wicket",
    palette="viridis"
)
plt.title("Top 10 Bowlers by Wickets in IPL")
plt.xlabel("Bowlers")
plt.ylabel("Wickets")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("top_bowlers.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
#best economical bowler
balls_bowled=data.groupby("bowler")["ball"].count().reset_index()
bowler_runs=data.groupby("bowler")["total_runs"].sum().reset_index()

bowler_stats=bowler_runs.merge(balls_bowled,on="bowler")
bowler_stats["overs"]=bowler_stats["ball"]/6
#economy calculate #
bowler_stats["economy"]=bowler_stats["total_runs"]/bowler_stats["overs"]
bowler_stats=bowler_stats.sort_values("economy").head(10).reset_index(drop=True)
bowler_stats

In [ ]:
#Visualization(Best Economical Bowlers)
plt.figure(figsize=(10,6))
sns.barplot(
    data=bowler_stats,
    x="bowler",
    y="economy",
    palette="Set1"
)
plt.title("Top 10 Economical Bowlers in IPL")
plt.xlabel("Bowler")
plt.ylabel("Economy Rate")
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig("economical_bowlers.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
#Team Performance
#Standardize Team Names
team_name_map= {
    "Royal Challengers Bangalore":"Royal Challengers Bengaluru",
    "Delhi Daredevils":"Delhi Capitals",
    "Kings XI Punjab":"Punjab Kings",
    "Rising Pune Supergiant":"Pune Supergiant",
    "Rising Pune Supergiants":"Pune Supergiant"
}
#Standardize batting and bowling team names
batting_team=df['batting_team'].replace(team_name_map)
bowling_team=df['bowling_team'].replace(team_name_map)

#match winners finding
match_scores=df.groupby(['match_id','batting_team'])['total_runs'].sum().reset_index()
match_scores['batting_team']=match_scores['batting_team'].replace(team_name_map)
match_winners=match_scores.loc[match_scores.groupby('match_id')['total_runs'].idxmax()]

#Total matches played per team (1 per team per match)
all_teams=pd.concat([
    df[['match_id']].assign(team=batting_team),
    df[['match_id']].assign(team=bowling_team)
]).drop_duplicates()
matches_played=all_teams.groupby('team')['match_id'].count().reset_index(name='matches_played')

#Total wins per team
wins=match_winners['batting_team'].value_counts().reset_index()
wins.columns=['team','wins']

#Merge matches and wins calculate losses
team_stats=matches_played.merge(wins,on='team',how='left').fillna(0)
team_stats['wins']=team_stats['wins'].astype(int)
team_stats['losses']=team_stats['matches_played']-team_stats['wins']

#team stats
team_stats

In [ ]:
#Team Stats Visualization
plt.figure(figsize=(14,10))
#bar plot
plt.subplot(2, 2, 1)
sns.barplot(data=team_stats, x='team', y='wins', palette='Set2')
plt.title("Wins per Team in IPL")
plt.xticks(rotation=45)
plt.xlabel("Team")
plt.ylabel("Wins")

#pie chart
plt.subplot(2, 2, 2)
plt.pie(team_stats['wins'], labels=team_stats['team'], autopct='%1.1f%%', startangle=140, colors=sns.color_palette('Set2'))
plt.title("Win Distribution Among Teams")

#line plot
plt.subplot(2, 2, 3)
sns.barplot(data=team_stats, x='team', y='matches_played', color='lightgray', label='Total Matches')
sns.lineplot(data=team_stats, x='team', y='wins', marker='o', color='green', linewidth=2, label='Wins')
plt.title("IPL Team Wins vs Total Matches")
plt.xlabel("Team")
plt.ylabel("Number of Matches / Wins")
plt.xticks(rotation=45)
plt.legend()

plt.tight_layout()
plt.savefig("team_stats_composite.png", dpi=300, bbox_inches='tight')
plt.show()